In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
EIS 单弧圆拟合处理脚本
- 读入类似 `30_degree_Real`, `30_degree_Imag` 的列
- 对每个温度做主半圆圆拟合，取左右截距的差作为 R
- 用 σ = L/(R*A) 计算电导率，输出 Arrhenius 拟合与 Ea

用法：
    python eis_circlefit_singlearc.py Li3N_combined_eis_data_01.csv ...
若不传参数，则会尝试处理当前目录下常见的 4 个文件名。
"""

import sys
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import lstsq
from scipy.signal import savgol_filter, find_peaks

# ======== 几何参数（按需修改） ========
SAMPLE_THICKNESS_CM = 0.002    # L
SAMPLE_AREA_CM2     = 0.079    # A
L_over_A = SAMPLE_THICKNESS_CM / SAMPLE_AREA_CM2

# ======== 基础工具函数 ========
def detect_temps(columns):
    """从列名里提取温度整数（如 30_degree_Real -> 30）。"""
    temps = []
    for c in columns:
        m = re.match(r'(\d+)_degree_Real$', str(c))
        if m:
            temps.append(int(m.group(1)))
    return sorted(set(temps))

def clean_xy(df, T):
    """
    取某温度的 Zre, Zim 列，转为数值，去 NaN，按 Zre 升序排序。
    Nyquist 采用 y = -Zim（向上）。
    """
    xr = pd.to_numeric(df[f"{T}_degree_Real"], errors="coerce").to_numpy()
    xi = pd.to_numeric(df[f"{T}_degree_Imag"], errors="coerce").to_numpy()
    mask = np.isfinite(xr) & np.isfinite(xi)
    x = xr[mask]
    y = -xi[mask]
    if len(x) == 0:
        return x, y
    idx = np.argsort(x)
    return x[idx], y[idx]

def fit_circle_kasa(x, y):
    """
    Kåsa 线性最小二乘圆拟合：
    (x-a)^2 + (y-b)^2 = r^2
    展开为：x^2 + y^2 + D x + E y + F = 0
    由最小二乘解得 D,E,F -> a=-D/2, b=-E/2, r=sqrt(a^2+b^2-F)
    """
    A = np.column_stack([x, y, np.ones_like(x)])
    bvec = -(x**2 + y**2)
    D, E, F = lstsq(A, bvec, rcond=None)[0]
    a = -D / 2.0
    b0 = -E / 2.0
    r_sq = a*a + b0*b0 - F
    if r_sq <= 0:
        return None
    r = np.sqrt(r_sq)
    return a, b0, r

def x_intercepts(a, b, r):
    """给定圆心(a,b)、半径r，求与 x 轴的交点（y=0）的 x 值（左、右）。"""
    disc = r*r - b*b
    if disc < 0:
        return None
    d = np.sqrt(disc)
    return (a - d, a + d)

def estimate_R_single_arc(x, y):
    """
    找主半圆（y 的主峰）。在峰顶附近截取窗口做圆拟合，返回左右截距差 R。
    """
    if len(x) < 25:
        return np.nan, np.nan, np.nan  # x_left, x_right, R

    # 平滑曲线，提取主峰
    window = 21 if len(y) >= 41 else max(7, (len(y)//5)*2+1)
    try:
        y_s = savgol_filter(y, window_length=window, polyorder=3)
    except Exception:
        y_s = y

    peaks, _ = find_peaks(y_s, prominence=0.01*np.nanmax(y_s) if np.nanmax(y_s) > 0 else 0.0)
    if len(peaks) == 0:
        pk = int(np.argmax(y_s))
    else:
        # 选最靠左（Zre 最小）的主峰
        pk = int(sorted(peaks, key=lambda i: x[i])[0])

    # 峰顶附近做圆拟合
    left = max(0, pk - 18)
    right = min(len(x), pk + 19)
    res = fit_circle_kasa(x[left:right], y[left:right])
    if res is None:
        return np.nan, np.nan, np.nan

    a, b0, r = res
    xi = x_intercepts(a, b0, r)
    if xi is None:
        return np.nan, np.nan, np.nan
    xL, xR = xi

    R = xR - xL if np.isfinite(xR) and np.isfinite(xL) else np.nan
    return xL, xR, R

def process_file(csv_path: Path):
    """处理单个 CSV，返回结果 DataFrame，并写出 CSV 与 Arrhenius 图。"""
    df = pd.read_csv(csv_path, header=0, low_memory=False)
    temps = detect_temps(df.columns.tolist())
    rows = []
    for T in temps:
        x, y = clean_xy(df, T)
        if len(x) == 0:
            continue
        xL, xR, R = estimate_R_single_arc(x, y)
        if not (np.isfinite(R) and R > 0):
            continue
        sigma = L_over_A / R
        TK = T + 273.15
        rows.append({
            "Temperature (°C)": T,
            "x_left (Ohm)": xL,
            "x_right (Ohm)": xR,
            "R_total (Ohm)": R,
            "Conductivity_total (S/cm)": sigma,
            "1000/T (K^-1)": 1000.0 / TK,
            "ln(σ_total)": np.log(sigma) if sigma > 0 else np.nan
        })

    res = pd.DataFrame(rows).sort_values("Temperature (°C)")
    return res

def save_arrhenius(res_df: pd.DataFrame, title_prefix: str, out_png: Path):
    """保存 Arrhenius 图并返回活化能 Ea（eV）。"""
    valid = res_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["ln(σ_total)", "1000/T (K^-1)"])
    if len(valid) < 2:
        return np.nan

    x = valid["1000/T (K^-1)"].to_numpy()
    y = valid["ln(σ_total)"].to_numpy()
    slope, intercept = np.polyfit(x, y, 1)
    # ln σ = ln σ0 - Ea/(kB * T)；此处 x=1000/T，因此 Ea = -slope * kB * 1000
    kB_eV = 8.617333262e-5  # eV/K
    Ea_eV = -slope * kB_eV * 1000.0

    plt.figure()
    plt.scatter(x, y)
    xs = np.linspace(x.min(), x.max(), 200)
    ys = slope * xs + intercept
    plt.plot(xs, ys)
    plt.xlabel("1000/T (K^-1)")
    plt.ylabel("ln(σ_total)")
    plt.title(f"{title_prefix} | Ea ≈ {Ea_eV:.3f} eV")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()
    return Ea_eV

def main():
    # 待处理文件列表：命令行参数；若为空，尝试这些常见名（存在才处理）
    args = [Path(a) for a in sys.argv[1:]]
    if not args:
        candidates = [
            "Li3N_combined_eis_data_01.csv",
            "HfO2_combined_eis_data_01.csv",
            "Sc-HfO2-Li3N_combined_eis_data.csv",
            "Y-HfO2-Li3N_combined_eis_data.csv",
        ]
        args = [Path(c) for c in candidates if Path(c).exists()]

    if not args:
        print("未指定 CSV，也未在当前目录找到示例文件。请把 CSV 路径作为参数传入。")
        sys.exit(1)

    for csv in args:
        if not csv.exists():
            print(f"[跳过] 找不到文件：{csv}")
            continue

        print(f"\n=== 处理：{csv.name} ===")
        res = process_file(csv)
        if res.empty:
            print("未得到有效温度点/主半圆，结果为空。")
            continue

        out_csv = csv.with_name(csv.stem + "_single_arc_fit_results.csv")
        res.to_csv(out_csv, index=False)
        Ea = save_arrhenius(res, csv.stem, csv.with_name(csv.stem + "_arrhenius.png"))

        # 简短摘要
        print(res[["Temperature (°C)", "R_total (Ohm)", "Conductivity_total (S/cm)"]]
              .to_string(index=False, float_format=lambda x: f"{x:.6e}"))
        if np.isfinite(Ea):
            print(f"活化能 Ea ≈ {Ea:.3f} eV")
        print(f"已导出：{out_csv.name} 及 {csv.stem}_arrhenius.png")

if __name__ == "__main__":
    main()
